In [ ]:
# Install required libraries only if your environment is missing them.
# Set RUN_INSTALLS = True, run this cell once, then restart the notebook kernel.
RUN_INSTALLS = False # Change to True if imports fail due to missing packages.

if RUN_INSTALLS:
    import subprocess
    import sys

    packages = [
        "gradio",
        "ultralytics",
        "opencv-python",
        "pillow",
        "pandas",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *packages])
else:
    print("Install step skipped. Set RUN_INSTALLS = True if imports fail.")


# 04. Model Deployment Using Gradio

This notebook loads the trained DhakaRoadNet YOLOv8 model and starts a simple Gradio app. Upload a real road image, run detection, and review the annotated output before moving to TensorFlow Lite and Android deployment.

In [17]:
from datetime import datetime
from pathlib import Path
import json
import time

import cv2
import gradio as gr
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

# Main settings. Change these values while testing real images.
MODEL_RELATIVE_PATH = Path("model/checkpoints/yolov8n_dhakaroadnet_baseline/best.pt")
CONFIDENCE = 0.25
IOU = 0.70
IMAGE_SIZE = 640
MAX_DETECTIONS = 300

# True gives a temporary public Gradio link. False gives local host only.
SHARE_PUBLIC_LINK = False


In [18]:
def find_project_root(model_relative_path: Path) -> Path:
    """Find the project root whether the notebook is run from root or notebooks/."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / model_relative_path).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {model_relative_path}. Run this notebook from the project, "
        "or check that training produced model/checkpoints/.../best.pt."
    )


PROJECT_ROOT = find_project_root(MODEL_RELATIVE_PATH)
MODEL_PATH = PROJECT_ROOT / MODEL_RELATIVE_PATH
OUTPUT_ROOT = PROJECT_ROOT / "reports" / "gradio_tests"
IMAGE_OUTPUT_DIR = OUTPUT_ROOT / "images"
LOG_OUTPUT_DIR = OUTPUT_ROOT / "logs"
IMAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

model = YOLO(str(MODEL_PATH))
class_names = model.names

print(f"Project root: {PROJECT_ROOT}")
print(f"Model path: {MODEL_PATH}")
print(f"Device: {DEVICE_NAME}")
print(f"Classes: {len(class_names)}")
print(f"Outputs will be saved to: {OUTPUT_ROOT}")


Project root: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI
Model path: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\checkpoints\yolov8n_dhakaroadnet_baseline\best.pt
Device: NVIDIA GeForce RTX 3050
Classes: 24
Outputs will be saved to: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\reports\gradio_tests


In [19]:
def prepare_image(image):
    if image is None:
        raise gr.Error("Please upload a road image first.")
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    return Image.fromarray(image).convert("RGB")


def result_to_table(result) -> pd.DataFrame:
    columns = ["class_id", "class_name", "confidence", "x1", "y1", "x2", "y2"]
    if result.boxes is None or len(result.boxes) == 0:
        return pd.DataFrame(columns=columns)

    boxes = result.boxes.xyxy.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    rows = []
    for class_id, confidence, box in zip(class_ids, confidences, boxes):
        x1, y1, x2, y2 = box.tolist()
        rows.append(
            {
                "class_id": int(class_id),
                "class_name": class_names.get(int(class_id), str(class_id)),
                "confidence": round(float(confidence), 4),
                "x1": round(float(x1), 2),
                "y1": round(float(y1), 2),
                "x2": round(float(x2), 2),
                "y2": round(float(y2), 2),
            }
        )
    return pd.DataFrame(rows, columns=columns).sort_values("confidence", ascending=False)


def save_prediction(input_image, annotated_image, detection_table, metadata):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    input_path = IMAGE_OUTPUT_DIR / f"{timestamp}_input.jpg"
    output_path = IMAGE_OUTPUT_DIR / f"{timestamp}_prediction.jpg"
    csv_path = LOG_OUTPUT_DIR / f"{timestamp}_detections.csv"
    json_path = LOG_OUTPUT_DIR / f"{timestamp}_metadata.json"

    input_image.save(input_path)
    Image.fromarray(annotated_image).save(output_path)
    detection_table.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    return input_path, output_path, csv_path, json_path


def predict_image(image, confidence, iou, image_size, max_detections):
    input_image = prepare_image(image)
    start_time = time.perf_counter()

    results = model.predict(
        source=np.array(input_image),
        conf=float(confidence),
        iou=float(iou),
        imgsz=int(image_size),
        max_det=int(max_detections),
        device=DEVICE,
        verbose=False,
    )

    elapsed_seconds = time.perf_counter() - start_time
    result = results[0]
    annotated_bgr = result.plot()
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    detection_table = result_to_table(result)

    metadata = {
        "model_path": str(MODEL_PATH),
        "device": DEVICE_NAME,
        "confidence": float(confidence),
        "iou": float(iou),
        "image_size": int(image_size),
        "max_detections": int(max_detections),
        "detection_count": int(len(detection_table)),
        "elapsed_seconds": round(float(elapsed_seconds), 4),
    }
    _, output_path, csv_path, _ = save_prediction(input_image, annotated_rgb, detection_table, metadata)

    status = (
        f"Device: {DEVICE_NAME} | Detections: {len(detection_table)} | "
        f"Time: {elapsed_seconds:.2f}s | Saved: {output_path.name}, {csv_path.name}"
    )
    return annotated_rgb, detection_table, status


In [22]:
with gr.Blocks(title="DhakaRoadNet Road Object Detection") as demo:
    gr.Markdown("# DhakaRoadNet Road Object Detection")
    gr.Markdown("Upload a real road image and test the trained YOLOv8 model before TFLite export.")

    with gr.Row():
        input_image = gr.Image(type="pil", label="Upload road image")
        output_image = gr.Image(type="numpy", label="Model prediction")

    with gr.Accordion("Prediction settings", open=False):
        confidence_slider = gr.Slider(0.05, 0.95, value=CONFIDENCE, step=0.05, label="Confidence threshold")
        iou_slider = gr.Slider(0.10, 0.90, value=IOU, step=0.05, label="IoU threshold")
        image_size_slider = gr.Slider(320, 1280, value=IMAGE_SIZE, step=32, label="Image size")
        max_detections_slider = gr.Slider(1, 500, value=MAX_DETECTIONS, step=1, label="Maximum detections")

    run_button = gr.Button("Run detection", variant="primary")
    detection_table = gr.Dataframe(label="Detected objects", interactive=False)
    status_text = gr.Textbox(label="Status", interactive=False)

    run_button.click(
        fn=predict_image,
        inputs=[input_image, confidence_slider, iou_slider, image_size_slider, max_detections_slider],
        outputs=[output_image, detection_table, status_text],
    )

demo.launch(
    server_name="127.0.0.1",
    server_port=7870,
    share=SHARE_PUBLIC_LINK,
)


* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
